# Other-Topics Area Classifier & Pairing Playground

Takes a **pointwise** or **pairwise** YAML filtered to *"other topics in machine learning"*,
sends all abstracts to an LLM in a **single call** that simultaneously assigns a specific
research area to each abstract and groups them into matched pairs.

Typical input: output of `scripts/data_stats.py --extract_other`.

## Setup

In [ ]:
import sys
import os
import json
import re
from pathlib import Path
from IPython.display import display, HTML
import yaml
import toml

# ── paths ─────────────────────────────────────────────────────────────────────
SRC_ROOT  = Path("../../..").resolve()   # → .../Academic-Collabs/src
REPO_ROOT = SRC_ROOT.parent
sys.path.insert(0, str(SRC_ROOT))

# ── secrets ───────────────────────────────────────────────────────────────────
_secrets_path = REPO_ROOT / "secrets.toml"
assert _secrets_path.exists(), f"secrets.toml not found at {_secrets_path}"
_secrets = toml.load(_secrets_path)

os.environ["SECRETS"]           = str(_secrets_path)
os.environ["OUTPUT_DIR"]        = str(REPO_ROOT / "output")
os.environ["OPENAI_API_KEY"]    = _secrets.get("openai_key", "")
os.environ["ANTHROPIC_API_KEY"] = _secrets.get("anthropic", "")

from utils import prompt_openai_client

print(f"SRC_ROOT : {SRC_ROOT}")
print(f"REPO_ROOT: {REPO_ROOT}")

---
## Configuration

In [ ]:
# ── Input file (pointwise or pairwise YAML filtered to 'other topics') ────────
INPUT_YAML = REPO_ROOT / "output/baseline-test-data/iclr-vs-iclr/manipulated-pointwise-20260501_123408/iclr_pointwise_instances_other_topics.yaml"

# ── Model ─────────────────────────────────────────────────────────────────────
# OpenAI  : "gpt-4o", "o3"
# Anthropic: "claude-sonnet-4-5", "claude-opus-4-5"
MODEL = "claude-opus-4-7"

---
## Load data

In [ ]:
with open(INPUT_YAML) as f:
    raw = yaml.safe_load(f)

def _is_pairwise(data: dict) -> bool:
    first = next(iter(data.values()))
    return "ideas" in first and isinstance(first.get("metadata"), dict)

ideas: list[dict] = []

if _is_pairwise(raw):
    fmt = "pairwise"
    for pair_key, pair in raw.items():
        for idea_key, abstract in pair["ideas"].items():
            meta = pair["metadata"].get(idea_key, {})
            ideas.append({
                "id": f"{pair_key}:{idea_key}",
                "abstract": abstract,
                "title": meta.get("title", ""),
                "label": meta.get("type", "?"),
                "rating": meta.get("rating", "?"),
            })
else:
    fmt = "pointwise"
    for key, entry in raw.items():
        meta = entry.get("metadata", {})
        ideas.append({
            "id": str(key),
            "abstract": entry.get("idea", ""),
            "title": meta.get("title", ""),
            "label": entry.get("label", "?"),
            "rating": meta.get("rating", "?"),
        })

print(f"Format : {fmt}")
print(f"Ideas  : {len(ideas)}")
for idea in ideas:
    print(f"  [{idea['id']:>6}]  {idea['label']:8}  {idea['title'][:70]}")

---
## Classify areas and match pairs — single LLM call

In [ ]:
def build_prompt(ideas: list[dict]) -> str:
    abstracts_block = "\n\n".join(
        f'ID: {idea["id"]} | Label: {idea["label"]}\nAbstract: {idea["abstract"]}'
        for idea in ideas
    )
    return f"""\
You are a machine learning research expert.
Below are {len(ideas)} paper abstracts, each with an ID.

Your task:
1. Group the IDs into negative-positive pairs based on shared or closely related research topics. Singletons are allowed when no reasonable match exists.
2. For each pair (or singleton), assign ONE umbrella area label (3–7 words) that best describes the shared topic of the group.

Important:
* There should be 1 positive and 1 negative in each pair. If you can't find a good positive-negative match, it's better to leave them as a singleton.
* Multiple pairs can share the same area label if they fit under the same umbrella.
* You should aspire to produce a conference style (e.g., ICLR, ACL) area names.
* Avoid compound area names that simply list the two topics of the paired abstracts (e.g., "Uncertainty Estimation and Active Learning"). The area should apply to both abstracts.

Return ONLY valid JSON in exactly this schema — no prose, no markdown fences:
{{
  "pairs": [
    {{"ids": ["<id>", "<id>"], "labels": [<pos/neg>, <pos/neg>], "area": "<shared umbrella area>"}},
    ...
  ]
}}

Abstracts:
{abstracts_block}
"""


def extract_json(text: str) -> dict:
    """Find the outermost {...} by tracking brace depth, then parse it."""
    start = text.find('{')
    if start == -1:
        raise ValueError(f"No JSON object found in response:\n{text}")
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise ValueError(f"Unbalanced braces in response:\n{text}")


print(f"Sending {len(ideas)} abstracts to {MODEL}...")
prompt = build_prompt(ideas)
raw_response = prompt_openai_client(prompt, engine=MODEL, max_completion_tokens=1024)
print("Raw response:")
print(raw_response)

In [ ]:
result = extract_json(raw_response)

# pairs is now: [{"ids": [...], "area": "..."}, ...]
pairs: list[dict] = result["pairs"]

id_map = {idea["id"]: idea for idea in ideas}

print(f"Groups ({len(pairs)}):")
for g_idx, pair in enumerate(pairs):
    members = [id_map[i] for i in pair["ids"] if i in id_map]
    titles  = " / ".join(m["title"][:45] for m in members)
    print(f"  Group {g_idx}: {pair['ids']}  →  [{pair['area']}]  {titles}")

---
## Results

In [ ]:
LABEL_COLORS  = {"POSITIVE": "#2a7a2a", "NEGATIVE": "#8b1a1a"}
GROUP_PALETTE = [
    "#3a6186", "#7b4f8e", "#2e7d5e", "#7a5c1e",
    "#1a5c7a", "#6b2d5e", "#3d6b3a", "#7a3d1e",
]

def _idea_box(idea: dict, accent: str) -> str:
    label_color = LABEL_COLORS.get(idea['label'], '#555')
    return f"""
    <div style="
        border:1px solid rgba(255,255,255,0.15);
        border-left:5px solid {accent};
        border-radius:6px;
        padding:14px 18px;
        margin:6px 0;
        line-height:1.7;
    ">
      <div style="margin-bottom:8px; display:flex; align-items:center; gap:8px; flex-wrap:wrap;">
        <span style="background:{accent}; color:#fff; font-family:monospace; font-size:13px;
                     padding:2px 8px; border-radius:4px;">ID {idea['id']}</span>
        <span style="background:{label_color}; color:#fff; font-family:monospace; font-size:12px;
                     padding:2px 6px; border-radius:4px;">{idea['label']}</span>
        <span style="font-size:13px; opacity:0.7;">rating: {idea['rating']}</span>
      </div>
      <div style="font-weight:600; font-size:15px; margin-bottom:6px;">{idea['title']}</div>
      <div style="font-size:15px;">{idea['abstract']}</div>
    </div>
    """

for g_idx, pair in enumerate(pairs):
    members = [id_map[i] for i in pair["ids"] if i in id_map]
    if not members:
        continue
    accent = GROUP_PALETTE[g_idx % len(GROUP_PALETTE)]

    display(HTML(f"""
    <div style="padding:10px 18px; border-radius:8px; border-left:6px solid {accent};
                font-family:sans-serif; font-size:16px;
                margin-top:28px; margin-bottom:6px; opacity:0.9;">
        <span style="opacity:.6; font-size:13px;">
            GROUP {g_idx} &nbsp;·&nbsp; {len(members)} idea{'s' if len(members) != 1 else ''}
        </span><br/>
        <b>{pair['area']}</b>
    </div>
    """))

    for idea in members:
        display(HTML(_idea_box(idea, accent)))

    display(HTML("<hr style='border:none;border-top:1px solid rgba(128,128,128,0.2);margin:12px 0'/>"))

---
## Export pairs

In [ ]:
pairs_out = {}
pair_idx  = 0

for pair in pairs:
    members = [id_map[i] for i in pair["ids"] if i in id_map]
    for i in range(0, len(members) - 1, 2):
        a, b   = members[i], members[i + 1]
        winner = 0 if a['label'] == 'POSITIVE' else 1
        pairs_out[str(pair_idx)] = {
            "context":          pair["area"],
            "ideas":            {0: a["abstract"], 1: b["abstract"]},
            "expected_winners": [winner],
            "metadata": {
                0: {"title": a["title"], "type": a["label"], "rating": a["rating"]},
                1: {"title": b["title"], "type": b["label"], "rating": b["rating"]},
            },
        }
        pair_idx += 1

out_path = Path(INPUT_YAML).parent / (Path(INPUT_YAML).stem + "_pairs.yaml")
with open(out_path, "w") as f:
    yaml.dump(pairs_out, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

print(f"Exported {pair_idx} pair(s) → {out_path}")